# Slide 13: Glue-based data lineage from source to consumption

**Classroom story:** A raw policy table is loaded into the AWS Glue Data Catalog, transformed by a Glue ETL job, and then consumed by a downstream application. The lineage story is: source table -> Glue ETL job -> curated table -> consumer answer.

This notebook creates a demo of the same lineage pattern that AWS Glue tracks in Glue Studio. It records a lineage manifest with the source table, transformation run, curated output, and consumption event. This is a teaching demo of how Glue lineage is represented, not a replacement for the actual AWS Glue console lineage view.

**Run in VS Code:** select a Python 3 notebook kernel and run the cells in order. The first cells use only the Python standard library so they work locally. The Glue ETL snippets are prepared as examples for a real Glue job or Glue Studio job script.

**Trainer question before running:** “If the final answer is wrong, can we identify the original source record, the transformation version, and the job run that produced it?”

## 1. Source: raw policy data in the Glue Data Catalog

In a real Glue demo, the source is a table in the AWS Glue Data Catalog. For this notebook, we simulate that source as a raw CSV with a stable business key so learners can see the lineage path clearly. In production, that row would come from a crawler-managed table or a manually registered table in Glue.


In [ ]:
from pathlib import Path
import csv, hashlib, json, uuid
from datetime import datetime, timezone

work = Path.cwd() / "lineage_demo"
work.mkdir(exist_ok=True)
raw_path = work / "policies_raw.csv"
rows = [
    {"policy_id":"P100", "updated_at":"2026-08-01", "owner_email":"maya@example.com",
     "policy_text":"Staff may carry forward up to 5 leave days."},
    {"policy_id":"P200", "updated_at":"2026-08-10", "owner_email":"arun@example.com",
     "policy_text":"Travel claims must be submitted within 30 days."},
]
with raw_path.open("w", newline="", encoding="utf-8") as f:
    writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
def sha256(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
run_id=uuid.uuid4().hex
source_hash=sha256(raw_path)
print("Source:",raw_path)
print(raw_path.read_text(encoding="utf-8"))
print("Source SHA-256:",source_hash)
print("Run ID:",run_id)


## 2. Transform: Glue ETL job validates and cleans the source

This is the Glue-equivalent step: a source table is read from the Glue Data Catalog, validated, and transformed into a curated table. The notebook writes a manifest describing **source table -> Glue ETL job -> curated output -> run ID**. In the actual AWS Glue console, this lineage graph is shown automatically when the job runs and the Data Catalog tables are linked correctly.


A real AWS Glue ETL job (Spark) for this pattern usually looks like this:

```python
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.context import SparkContext

sc = SparkContext()
glue_context = GlueContext(sc)
spark = glue_context.spark_session
job = Job(glue_context)
job.init("policy_lineage_demo", {"--JOB_NAME": "policy_lineage_demo"})

# Read the source table from the Glue Data Catalog
source_df = glue_context.create_dynamic_frame.from_catalog(
    database="raw_db",
    table_name="policies_raw",
)

# Example transformation: keep required fields and drop the personal email column
clean_df = source_df.select_fields(["policy_id", "updated_at", "policy_text"])

# Write the curated table back to the Data Catalog
glue_context.write_dynamic_frame.from_catalog(
    frame=clean_df,
    database="curated_db",
    table_name="policies_curated",
)

job.commit()
```

The local cells below reproduce the same source -> transform -> curated -> consume path with the Python standard library, so learners can trace lineage without provisioning a Glue job.

In [ ]:
TRANSFORM_VERSION="policy-clean-v1"
approved_path=work/"policies_approved.jsonl"
manifest_path=work/"lineage_manifest.json"
with raw_path.open(newline="",encoding="utf-8") as f:
    input_rows=list(csv.DictReader(f))
ids=[r["policy_id"] for r in input_rows]
assert len(ids)==len(set(ids)) and all(ids), "Source IDs must be present and unique"
approved=[];record_lineage={}
for line_no,r in enumerate(input_rows,start=2):
    record={"document_id":r["policy_id"],"text":r["policy_text"],"source_updated_at":r["updated_at"]}
    approved.append(record)
    record_lineage[record["document_id"]]={"source_row":line_no,"source_policy_id":r["policy_id"],
        "source_sha256":source_hash,"transformation":TRANSFORM_VERSION,"run_id":run_id}
with approved_path.open("w",encoding="utf-8") as f:
    for r in approved:f.write(json.dumps(r,ensure_ascii=False)+"\n")
manifest={"run_id":run_id,"created_at":datetime.now(timezone.utc).isoformat(),
    "source":{"uri":raw_path.as_uri(),"sha256":source_hash},
    "transformation":{"name":"validate_and_redact","version":TRANSFORM_VERSION,
        "checks":["nonempty unique IDs"],"removed_columns":["owner_email"]},
    "output":{"uri":approved_path.as_uri(),"sha256":sha256(approved_path)},
    "record_lineage":record_lineage,"consumption_events":[]}
manifest_path.write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("Approved content:")
print(approved_path.read_text(encoding="utf-8"))
print("Manifest:",manifest_path)


## 3. Consumption: downstream system reads the curated data

This is the consumer layer in a Glue lineage story: the curated table is read by a downstream process, and the application records which document or row it used. In a real AWS architecture, this could be a reporting job, a query in Athena, a model training job, or a serverless API reading from the curated data source.


The log below simulates a downstream consumption event so the lineage chain remains visible: source table -> Glue ETL job -> curated table -> consumer question/answer.

In [ ]:
question="How many leave days can staff carry forward?"
matches=[r for r in approved if "leave days" in r["text"].lower()]
assert len(matches)==1
used=matches[0]
answer=used["text"]
event={"request_id":uuid.uuid4().hex,"timestamp":datetime.now(timezone.utc).isoformat(),
    "question":question,"answer":answer,"consumed_document_ids":[used["document_id"]],
    "pipeline_run_id":run_id}
manifest["consumption_events"].append(event)
manifest_path.write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("Question:",question)
print("Answer:",answer)
print("Citation document ID:",used["document_id"])
print("Request ID:",event["request_id"])


## 4. Trace backward from the answer to the Glue source

Tell learners: “I start with the consumer request, find the curated output record it used, then follow the transformation run back to the original Glue source table and row.” The next cell proves each hop and checks that the file hashes still match.


In [ ]:
saved=json.loads(manifest_path.read_text(encoding="utf-8"))
consumption=next(x for x in saved["consumption_events"] if x["request_id"]==event["request_id"])
document_id=consumption["consumed_document_ids"][0]
origin=saved["record_lineage"][document_id]
assert saved["source"]["sha256"]==sha256(raw_path),"Source bytes changed"
assert saved["output"]["sha256"]==sha256(approved_path),"Approved bytes changed"
with raw_path.open(newline="",encoding="utf-8") as f:
    source_row=list(csv.DictReader(f))[origin["source_row"]-2]
assert source_row["policy_id"]==origin["source_policy_id"]
print("CONSUMPTION:",consumption["request_id"],"answered using",document_id)
print("OUTPUT:",saved["output"]["uri"])
print("TRANSFORM:",origin["transformation"],"run",origin["run_id"])
print("SOURCE:",saved["source"]["uri"],"row",origin["source_row"])
print("SOURCE CONTENT:",source_row["policy_text"])
print("SHA-256 checks: PASSED")


## 5. Impact analysis: what must be refreshed when the Glue source changes?

If the source table is updated, then the curated tables and any consumer answers derived from that source become stale until the job is rerun. Change the source policy for `P100` from **5** to **7** days in the first cell, then run only the cell below. It reports stale lineage. After that, rerun the source, transform, consumption, and trace cells in order to create a fresh lineage run. This is the concrete value of lineage: you can prove which downstream assets depend on which source version.


In [ ]:
expected=saved["source"]["sha256"]
actual=sha256(raw_path)
print("Source modified since approved output?", actual!=expected)
if actual!=expected:
    impacted=[k for k,v in saved["record_lineage"].items() if v["source_sha256"]==expected]
    print("Review/rebuild affected output IDs:",impacted)
    print("Review associated consumer answers:",
          [e["request_id"] for e in saved["consumption_events"]
           if set(e["consumed_document_ids"]) & set(impacted)])
else:
    print("Try a source edit to show downstream impact.")


## Optional: upload the evidence to S3

Supply an **existing** bucket in the selected Region. The notebook principal needs `s3:PutObject` on the demo prefix, `s3:HeadObject` to inspect object metadata, and `sts:GetCallerIdentity`. This cell uploads only synthetic classroom files. It writes under a unique run prefix, so a later run does not overwrite this evidence. If bucket versioning is enabled, S3 also returns `VersionId`. In the S3 console, open the three objects and inspect their metadata and versions. The manifest still contains local file URIs; for a production pipeline, record the S3 URI and version ID in the manifest at creation time.


In [ ]:
UPLOAD_TO_S3=False
AWS_PROFILE=None  # None uses the standard credential chain; or set a named profile.
AWS_REGION="<AWS_REGION>"  # e.g. "ap-south-1"
BUCKET=""
if UPLOAD_TO_S3:
    import boto3
    if not BUCKET:raise ValueError("Set BUCKET to an existing training bucket")
    session=boto3.Session(profile_name=AWS_PROFILE,region_name=AWS_REGION)
    s3=session.client("s3")
    prefix=f"lineage-demo/{run_id}/"
    for p in (raw_path,approved_path,manifest_path):
        key=prefix+p.name
        s3.upload_file(str(p),BUCKET,key,ExtraArgs={"Metadata":{"pipeline-run-id":run_id,"sha256":sha256(p)}})
        head=s3.head_object(Bucket=BUCKET,Key=key)
        print(f"s3://{BUCKET}/{key}","version:",head.get("VersionId","bucket versioning off"),
              "hash:",head["Metadata"]["sha256"][:16]+"...")
else:print("Optional S3 upload disabled; all lineage evidence is available locally.")


## Trainer wrap-up

**Show the path:** Glue source table → Glue ETL job → curated table → downstream consumer answer → cited document ID. Then trace backward from the answer to the source record and run ID. Ask: “If a user disputes the answer, what evidence tells us whether the source was outdated or the transformation logic was wrong?”

**Key distinction:** AWS Glue lineage is captured by the Glue catalog and job metadata, while CloudTrail records AWS API calls. Neither alone fully explains every business-level source-to-consumption relationship in a custom application. This notebook records the lineage links explicitly so learners can see the reasoning path. In production, the recommended pattern is to use Glue Studio and the Glue Data Catalog to visualize the lineage graph, then enrich it with custom application-level provenance when needed.

**AWS reading:** [AWS Glue Data Catalog](https://docs.aws.amazon.com/glue/latest/dg/catalog-and-crawler.html), [AWS Glue Studio](https://docs.aws.amazon.com/glue/latest/dg/console-studio.html), [AWS Glue ETL](https://docs.aws.amazon.com/glue/latest/dg/aws-glue-programming-etl.html), [Amazon DataZone user guide](https://docs.aws.amazon.com/datazone/latest/userguide/what-is-datazone.html).
